In [ ]:
import torch
import zipfile
import os
import decimal as d
import numpy as np
import json
import unet
import image_processor
import torchvision
from datetime import datetime
import matplotlib.pyplot as plt
import itertools
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression, LinearRegression
import pickle
import fnmatch

In [ ]:
def modelLoader(model, path: str):
    return model.load_state_dict(torch.load(path))

#thanks, Chat GPT
def unzip_file(zip_file_path, extract_to_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to_path)

#thanks, Chat GPT
def sortFirstNFiles(directoryPath: str, n: int = 10):
    allFiles = [os.path.join(directoryPath, file) for file in os.listdir(directoryPath) if file.endswith(('pt', '.pt'))]
    #sorts filenames based on third segment (validationloss in name)
    sortedFiles = sorted(allFiles, key = lambda fileName: int(fileName.split(os.sep)[-1].split('_')[2]))

    firstFiles = sortedFiles[:n]

    return firstFiles

def mapImageTensor(tensor: torch.Tensor, center = 0, scale = 1):
    return 1/ (1 + torch.exp(-(tensor - center)/scale))

def highPrecisionArray(numOfSamples, beg=0, last=1, precision = 4):
    d.getcontext().prec = precision
    start = d.Decimal(beg)
    end = d.Decimal(last)
    step = (end - start) / d.Decimal(numOfSamples - 1)
    numArray = [start + i * step for i in range(numOfSamples)]
    numArray = [float(x) for x in numArray]
    return numArray

def metrics(pred, label):
    if torch.is_tensor(pred) or torch.is_tensor(label):
        pred = pred.detach().cpu().numpy()
        label = label.detach().cpu().numpy()
    
    TP = ((label == 255) & (pred == 255)).sum()
    FP = ((label == 0) & (pred == 255)).sum()
    TN = ((label == 0) & (pred == 0)).sum()
    FN = ((label == 255) & (pred == 0)).sum()

    return int(TP), int(FP), int(TN), int(FN)

def dice(TP, FP, FN):
    if (2*TP + FP + FN) == 0:
        return 0
    return round(2*TP / ( 2*TP + FP + FN), 3)

def F1(TP, FP, FN):
    if (TP + .5*(FP + FN)) == 0:
        return 0
    return round(TP/ (TP + .5*(FP + FN)))

#thanks, Chat GPT
def rectTriArea(x_values, y_values):
    n = len(x_values)
    area = 0.0
    for i in range(1, n):
        width = x_values[i] - x_values[i - 1]
        height = min(y_values[i - 1], y_values[i])
        rectArea = width*height

        deltaHeight = max(y_values[i], y_values[i-1]) - height

        triArea = .5 * deltaHeight*width
        area += rectArea + triArea
    return area

def framePlotter(pred , label, frameNum: int, caseNum: int, modelName: str, saveDir: str):
    if torch.is_tensor(pred) or torch.is_tensor(label):
        pred = pred.detach().cpu().numpy()
        label = label.detach().cpu().numpy()
    plt.subplot(2,1,1)
    plt.imshow(pred)
    #parts = modelName.split("\\")
    #modelName = parts[3]
    plt.title(f'Pred- {modelName} | C: {caseNum} F: {frameNum}')

    plt.subplot(2,1,2)
    plt.imshow(label)
    plt.title(f'Label- {modelName} | C: {caseNum} F: {frameNum}')
    savePath = os.path.join(saveDir, modelName + '_C_' + str(caseNum) + '_F_' + str(frameNum) + '.png')
    plt.savefig(savePath)
    plt.show()

def findLogModels(logModelDir, modelName):
    matches = []
    for root, dirnames, filenames in os.walk(logModelDir):
        for filename in filenames:
            if fnmatch.fnmatch(filename, f'*{modelName}*'):
                matches.append(os.path.join(root, filename))
    return matches

def sigmoidalCurve(x, a, s, c):
    return a / (1 + np.exp(-s * (x - c)))

if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

In [ ]:


degree = 1
downscale = 16
testCases = 4
imagesPerTestCase = 100
numOfModelsToTest = 1
height = int(256/downscale)
width = int(1024/downscale)
shape = (height, width)

logModelDir = r""
modelDirectory = r""

thresholds = highPrecisionArray(61, 0, 1, 5)

transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((height, width))])


logModelImagePath = r""
logModelLabelPath = r""
segmentationPath = f"{logModelDir}\Segmentations"

if not os.path.exists(segmentationPath):
    os.makedirs(segmentationPath)

testImagePath = r""
testLabelPath = r""

topModelPaths = sortFirstNFiles(modelDirectory, numOfModelsToTest)

logModelFitLoader= image_processor.imageDirsToTestLoader(imageDir=logModelImagePath, labelDir= logModelLabelPath, transform=transform)
testLoader = image_processor.imageDirsToTestLoader(imageDir=testImagePath, labelDir=testLabelPath, transform=transform)

for modelPath in topModelPaths:
    currentTime = datetime.now()
    formatTime = currentTime.strftime("%Y-%m-%d_%H-%M-%S")
    print(f'START: {modelPath} | {formatTime}')
    print(f'Starting Log Model Loader')
    model = unet.unet()
    model.loadAttributes(modelPath)
    model.to(device)
    model.eval()

    listOfFlattenedOutputs = []
    listOfLabels = []

    for i, (image, label) in enumerate(logModelFitLoader):
        #with batch size of 1, i ranges from 0-399 in test set of 400 images
        #0-879 for training set
        image = image.to(device)
        label = label.to(device)
        output = model(image)
        #output = mapImageTensor(output)
        outputArray = output.detach().cpu().numpy().reshape(-1)
        labelArray = label.detach().cpu().numpy().reshape(-1)

        listOfFlattenedOutputs.append(outputArray)
        listOfLabels.append(labelArray)
    
    arrayOfFlattenedOutputs = np.array(listOfFlattenedOutputs).reshape(-1,1)
    poly = PolynomialFeatures(degree = degree)
    #each column in input is a 
    #880, 525825
    #transformedArrayOfFlattenedOutputs = poly.fit_transform(arrayOfFlattenedOutputs)
    #880, 1024
    arrayOfLabels = np.array(listOfLabels).reshape(-1)

    logModel = LogisticRegression().fit(arrayOfFlattenedOutputs, arrayOfLabels)


    
    probabilities = logModel.predict_proba(arrayOfFlattenedOutputs)

    bestDiceScore = 0
    bestF1Score = 0
    bestThreshold = 0

    currentTime = datetime.now()
    formatTime = currentTime.strftime("%Y-%m-%d_%H-%M-%S")
    print(f'START: Threshold Search | {formatTime}')

    for i, t in enumerate(thresholds):
        currentTime = datetime.now()
        formatTime = currentTime.strftime("%Y-%m-%d_%H-%M-%S")
        print(f'START: Threshold: {t} | {formatTime}')
        yPred = (probabilities[:, 1] > t).astype(int)
        yPredArray = np.array(yPred)
        yPredArray[yPredArray > .9] = 255
        TP, FP, TN, FN = metrics(yPredArray, arrayOfLabels)
        diceScore = dice(TP, FP, FN)
        #F1Score = F1(TP, FP, FN)
        if diceScore >= bestDiceScore :
            bestDiceScore = diceScore
            bestThreshold = t
    
    logModelName = f'{formatTime}_logModel_{model.name}_degree_{degree}_dice_{bestDiceScore}_F1_{bestF1Score}_t_{bestThreshold}'
    logModelPath = os.path.join(logModelDir, f'{logModelName}.pkl')

    with open(logModelPath, 'wb') as file:
        pickle.dump(logModel, file)



In [ ]:


for modelPath in topModelPaths:
    currentTime = datetime.now()
    formatTime = currentTime.strftime("%Y-%m-%d_%H-%M-%S")
    print(f'START: {modelPath} | {formatTime}')
    print(f'Starting Log Model Loader')
    model = unet.unet()
    model.loadAttributes(modelPath)
    model.to(device)
    model.eval()
    
    poly = PolynomialFeatures(degree = degree)
    matches = findLogModels(logModelDir, model.name)

    for match in matches:
        print(f'Analyzing {model.name} on Log Model {match}')
        with open(match, 'rb') as file:
            logModel = pickle.load(file)
        parts = match.split('_')
        parts = parts[-1].split('.pkl')
        t = float(parts[0])
        diceByFrame = [0]*100
        caseNum = 0
        
        #done with a batch size of one to not stress equipment
        for i, (image, label) in enumerate(testLoader):
            #with batch size of 1, i ranges from 0-399 in test set of 400 images
            #0-779 for training set
            image = image.to(device)
            label = label.to(device)
            output = model(image)
            #output = mapImageTensor(output)
            #(1024,)
            outputArray = output.detach().cpu().numpy().reshape(-1,1)
            #(256/downscale, 1024/downscale)
            labelArray = label.detach().cpu().numpy().reshape(shape)

            #arrayOfFlattenedOutputs = np.array(listOfFlattenedOutputs).reshape(-1,1)
            #arrayOfLabels = np.array(listOfLabels).reshape(-1)
            #transformedArrayOfFlattenedOutputs = poly.fit_transform(outputArray)
            probabilities = logModel.predict_proba(outputArray)
            yPred = (probabilities[:, 1] > t).astype(int)
            yPredArray = np.array(yPred).reshape(shape)
            yPredArray[yPredArray > .9] = 255


            TP, FP, TN, FN = metrics(yPredArray, labelArray)
            diceScore = dice(TP, FP, FN)

            #reshaping flattened array to graph predictions
            yPredArray.reshape(shape)
            labelArray.reshape(shape)

            frameNum = i%100
            if frameNum == 0:
                caseNum += 1

            diceByFrame[frameNum] += diceScore
            framePlotter(pred=yPredArray, label=labelArray, frameNum=frameNum, caseNum=caseNum, modelName= model.name, saveDir=segmentationPath)

            

        diceByFrame = list(map(lambda x: x * .25, diceByFrame))

        plt.plot(diceByFrame)
        plt.title(f"Avg. Dice by Frame for {model.name}")
        plt.show()




